### Reading parquet data with an inferred schema

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = (
    SparkSession.builder
    .appName("nested-dataframe")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "512m")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/24 14:09:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = (spark.read.format("json")
     .option("multiple", "true")
     .load("../data/Stanford Question Answering Dataset.json"))

In [5]:
df.printSchema()

root
 |-- paragraphs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- context: string (nullable = true)
 |    |    |-- qas: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- answers: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- answer_start: long (nullable = true)
 |    |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- id: string (nullable = true)
 |    |    |    |    |-- question: string (nullable = true)
 |-- title: string (nullable = true)



In [13]:
df_exploded = (
    df.select("title", explode("paragraphs").alias("paragraphs"))
        .select(
            "title",
            col("paragraphs.context").alias("context"),
            explode(col("paragraphs.qas")).alias("questions")
        )
)

In [14]:
df_exploded.show()

+-------------+--------------------+--------------------+
|        title|             context|           questions|
+-------------+--------------------+--------------------+
|Super_Bowl_50|Super Bowl 50 was...|{[{177, Denver Br...|
|Super_Bowl_50|Super Bowl 50 was...|{[{249, Carolina ...|
|Super_Bowl_50|Super Bowl 50 was...|{[{403, Santa Cla...|
|Super_Bowl_50|Super Bowl 50 was...|{[{177, Denver Br...|
|Super_Bowl_50|Super Bowl 50 was...|{[{488, gold}, {4...|
|Super_Bowl_50|Super Bowl 50 was...|{[{487, "golden a...|
|Super_Bowl_50|Super Bowl 50 was...|{[{334, February ...|
|Super_Bowl_50|Super Bowl 50 was...|{[{133, American ...|
|Super_Bowl_50|Super Bowl 50 was...|{[{487, "golden a...|
|Super_Bowl_50|Super Bowl 50 was...|{[{133, American ...|
|Super_Bowl_50|Super Bowl 50 was...|{[{334, February ...|
|Super_Bowl_50|Super Bowl 50 was...|{[{177, Denver Br...|
|Super_Bowl_50|Super Bowl 50 was...|{[{355, Levi's St...|
|Super_Bowl_50|Super Bowl 50 was...|{[{403, Santa Cla...|
|Super_Bowl_50

In [15]:
df_array_distinct = (
    df_exploded.select("title", "context",
                      col("questions.id").alias("question_id"),
                      col("questions.question").alias("question_text"),
                      array_distinct("questions.answers").alias("answers"))
            )

In [16]:
df_array_distinct.show()

+-------------+--------------------+--------------------+--------------------+--------------------+
|        title|             context|         question_id|       question_text|             answers|
+-------------+--------------------+--------------------+--------------------+--------------------+
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team re...|[{177, Denver Bro...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team re...|[{249, Carolina P...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Where did Super B...|[{403, Santa Clar...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team wo...|[{177, Denver Bro...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|What color was us...|[{488, gold}, {52...|
|Super_Bowl_50|Super Bowl 50 was...|56be8e613aeaaa140...|What was the them...|[{487, "golden an...|
|Super_Bowl_50|Super Bowl 50 was...|56be8e613aeaaa140...|What day was the ...|[{334, February 7...|


### Large number of rows with explode 

In [18]:
(
    df_array_distinct.select(
        "title",
        "context",
        "question_text",
        col("answers").getItem(0).getField("text").alias("answer")
    ).show()
)

+-------------+--------------------+--------------------+--------------------+
|        title|             context|       question_text|              answer|
+-------------+--------------------+--------------------+--------------------+
|Super_Bowl_50|Super Bowl 50 was...|Which NFL team re...|      Denver Broncos|
|Super_Bowl_50|Super Bowl 50 was...|Which NFL team re...|   Carolina Panthers|
|Super_Bowl_50|Super Bowl 50 was...|Where did Super B...|Santa Clara, Cali...|
|Super_Bowl_50|Super Bowl 50 was...|Which NFL team wo...|      Denver Broncos|
|Super_Bowl_50|Super Bowl 50 was...|What color was us...|                gold|
|Super_Bowl_50|Super Bowl 50 was...|What was the them...|"golden anniversary"|
|Super_Bowl_50|Super Bowl 50 was...|What day was the ...|    February 7, 2016|
|Super_Bowl_50|Super Bowl 50 was...|What is the AFC s...|American Football...|
|Super_Bowl_50|Super Bowl 50 was...|What was the them...|"golden anniversary"|
|Super_Bowl_50|Super Bowl 50 was...|What does AFC st

### Nested data with null values 

In [19]:
(
    df_array_distinct.filter(col("answers").getItem(0).getField("text").isNotNull()
    ).show()
)

+-------------+--------------------+--------------------+--------------------+--------------------+
|        title|             context|         question_id|       question_text|             answers|
+-------------+--------------------+--------------------+--------------------+--------------------+
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team re...|[{177, Denver Bro...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team re...|[{249, Carolina P...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Where did Super B...|[{403, Santa Clar...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|Which NFL team wo...|[{177, Denver Bro...|
|Super_Bowl_50|Super Bowl 50 was...|56be4db0acb800140...|What color was us...|[{488, gold}, {52...|
|Super_Bowl_50|Super Bowl 50 was...|56be8e613aeaaa140...|What was the them...|[{487, "golden an...|
|Super_Bowl_50|Super Bowl 50 was...|56be8e613aeaaa140...|What day was the ...|[{334, February 7...|


### `array_contains()`

### `map_keys()` and `map_values()`

### `explode_outer()`

### `posexplode()`

In [20]:
spark.stop()